In [0]:
%python
# Cell 1: Validate Daily Account Summary Constraints
summary_df = spark.table("main_dev.daily_account_summary")

# Test 1: Ensure columns are never null
null_account_count = summary_df.filter("account_id IS NULL OR transaction_date IS NULL").count()
assert null_account_count == 0, f"DATA QUALITY FAILURE: Found {null_account_count} rows with null primary keys."

# Test 2: Enforce granular composite unique key constraint
total_rows = summary_df.count()
distinct_primary_keys = summary_df.select("account_id", "transaction_date").distinct().count()
assert total_rows == distinct_primary_keys, "DATA QUALITY FAILURE: Primary key grain explosion detected (duplicate account/date combinations)."

# Test 3: Check business rules logic
negative_debits = summary_df.filter("total_debit_amount < 0").count()
assert negative_debits == 0, f"DATA QUALITY FAILURE: Detected {negative_debits} rows with impossible negative debit totals."

print("ALL DATA QUALITY TESTS PASSED CLEANLY!")
